In [1]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

In [ ]:
# табличка с мероприятиями

df = pd.read_excel('../data/Лофты посвяты.xlsx')
df.columns = ['Дата', 'Тип', 'Минимальная цена', 'Средняя цена', 'Ссылки']
print("✅ Загружено строк:", len(df))
df.head()

In [ ]:
#преобразуем дату
df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')
df['Год'] = df['Дата'].dt.year

#чистим цены
for col in ['Минимальная цена', 'Средняя цена']:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = df[col].str.replace(r'[^\d\.\-]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

#удаляем строки без года или без средней цены
df_clean = df.dropna(subset=['Год', 'Средняя цена']).copy()
df_clean['Тип'] = df_clean['Тип'].astype(str).str.strip().str.lower()

#отдельные датафреймы для лофт и посвят
loft = df_clean[df_clean['Тип'] == 'лофт']
posvyat = df_clean[df_clean['Тип'] == 'посвят']

loft_yearly = loft.groupby('Год')['Средняя цена'].mean().reset_index()
loft_yearly['Тип'] = 'Лофт'
posvyat_yearly = posvyat.groupby('Год')['Средняя цена'].mean().reset_index()
posvyat_yearly['Тип'] = 'Посвят'

print(f"Всего записей: {len(df_clean)}")
print(f"Лофт: {len(loft)}, Посвят: {len(posvyat)}")
print("Годы:", sorted(df_clean['Год'].unique()))

Всего записей: 28
Лофт: 19, Посвят: 9
Годы: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


/tmp/ipykernel_3491/3324891439.py:2: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')


In [ ]:
# индексы лофтов и посвятов относительно прошлых (отношение друг к другу)

loft_1 = loft.sort_values('Дата').copy().reset_index(drop=True)
posvyat_1 = posvyat.sort_values('Дата').copy().reset_index(drop=True)

def dynamics(table):
    table['Динамика цен (мин.)'] = 100.0
    table['Динамика цен (средн.)'] = 100.0

    for i in range(1, len(table)):
        price_min_curr = table.loc[i, 'Минимальная цена']
        price_min_prev = table.loc[i-1, 'Минимальная цена']
        price_avg_curr = table.loc[i, 'Средняя цена']
        price_avg_prev = table.loc[i-1, 'Средняя цена']
        table.loc[i, 'Динамика цен (мин.)'] = price_min_curr/price_min_prev*100
        table.loc[i, 'Динамика цен (средн.)'] = price_avg_curr/price_avg_prev*100
    return table

loft_1 = dynamics(loft_1)
posvyat_1 = dynamics(posvyat_1)

print(loft_1)
print(posvyat_1)

         Дата   Тип  Минимальная цена  Средняя цена  \
0  2019-08-30  лофт               800    900.000000   
1  2019-12-25  лофт               750    875.000000   
2  2020-09-04  лофт               750    875.000000   
3  2021-04-09  лофт               900   1050.000000   
4  2021-09-03  лофт               750    875.000000   
5  2021-12-17  лофт               900   1050.000000   
6  2022-04-15  лофт               900   1050.000000   
7  2022-09-02  лофт               900   1050.000000   
8  2022-12-17  лофт               999   1149.500000   
9  2023-04-22  лофт              1000   1150.000000   
10 2023-09-02  лофт              1000   1150.000000   
11 2023-12-23  лофт              1100   1200.000000   
12 2024-04-20  лофт              1300   1350.000000   
13 2024-09-06  лофт              1300   1400.000000   
14 2025-02-14  лофт              1505   1605.000000   
15 2025-04-25  лофт              1500   1600.000000   
16 2025-09-05  лофт              1800   2000.000000   
17 2026-02

In [ ]:
# загружаем данные по ИПЦ

df_ipc = pd.read_excel('../data/ИПЦ.xlsx')
df_ipc['ИПЦ'] = df_ipc['ИПЦ'].astype(float)
df_ipc = df_ipc.dropna().reset_index(drop=True)

print(df_ipc)

## Блок 1. Индексы, инфляция и реальные цены

Здесь мы считаем **лофты** и **посвяты** отдельно: у рядов разная природа и разная динамика.

**Что считаем:**

- годовой индекс цены к предыдущему году  

$$
I_t = \frac{P_t}{P_{t-1}} \cdot 100
$$

- накопленный ИПЦ РФ  

- реальную цену события  

$$
P_t^{real} = \frac{P_t}{\prod_{s=2}^{t} \left(\frac{\text{ИПЦ}_s}{100}\right)}
$$

ИПЦ берём из внешнего макроисточника и используем как бенчмарк для сравнения.


In [ ]:
# индексы лофтов и посвятов по годам + ИПЦ + реальная цена лофтов и посвятов

required_event_cols = {'Год', 'Средняя цена'}
for name, table in [('loft_yearly', loft_yearly), ('posvyat_yearly', posvyat_yearly)]:
    missing = required_event_cols - set(table.columns)
    if missing:
        raise KeyError(f"В {name} не хватает колонок: {missing}")

if 'ИПЦ' not in df_ipc.columns:
    raise KeyError("В таблице df_ipc должна быть колонка 'ИПЦ'")

df_ipc_yearly = df_ipc.copy()

if 'Год' not in df_ipc_yearly.columns:
    year_candidates = [c for c in df_ipc_yearly.columns if 'год' in str(c).lower()]
    if year_candidates:
        df_ipc_yearly = df_ipc_yearly.rename(columns={year_candidates[0]: 'Год'})
    else:
        df_ipc_yearly = df_ipc_yearly.rename(columns={df_ipc_yearly.columns[0]: 'Год'})

df_ipc_yearly = df_ipc_yearly[['Год', 'ИПЦ']].dropna().copy()
df_ipc_yearly['Год'] = df_ipc_yearly['Год'].astype(int)
df_ipc_yearly = df_ipc_yearly.sort_values('Год').reset_index(drop=True)

def indexes_yearly(table, index_name):
    table = table.sort_values('Год').reset_index(drop=True).copy()
    table[index_name] = 100.0
    for i in range(1, len(table)):
        table.loc[i, index_name] = 100 * table.loc[i, 'Средняя цена'] / table.loc[i - 1, 'Средняя цена']
    return table

def add_macro_and_real_price(table, index_name):
    table = indexes_yearly(table, index_name)
    table = table.merge(df_ipc_yearly, on='Год', how='left')

    if table['ИПЦ'].isna().any():
        missing_years = table.loc[table['ИПЦ'].isna(), 'Год'].tolist()
        print(f"⚠️ Для лет {missing_years} нет ИПЦ — реальная цена для них не считается")

    table['Накопленный_ИПЦ'] = np.nan
    if len(table) > 0:
        table.loc[0, 'Накопленный_ИПЦ'] = 1.0
        for i in range(1, len(table)):
            if pd.notna(table.loc[i, 'ИПЦ']) and pd.notna(table.loc[i - 1, 'Накопленный_ИПЦ']):
                table.loc[i, 'Накопленный_ИПЦ'] = table.loc[i - 1, 'Накопленный_ИПЦ'] * (table.loc[i, 'ИПЦ'] / 100)

    table['реальная цена'] = table['Средняя цена'] / table['Накопленный_ИПЦ']
    return table

loft_index = add_macro_and_real_price(loft_yearly, 'индекс лофтов')
posvyat_index = add_macro_and_real_price(posvyat_yearly, 'индекс посвятов')

print("Лофты:")
display(loft_index)
print("Посвяты:")
display(posvyat_index)

Лофты:


,Год,Средняя цена,Тип,индекс лофтов,ИПЦ,Накопленный_ИПЦ,реальная цена
0,2019,887.500000,Лофт,100.000000,103.04,1.000000,887.500000
1,2020,875.000000,Лофт,98.591549,104.91,1.049100,834.048232
2,2021,991.666667,Лофт,113.333333,108.39,1.137119,872.086597
3,2022,1083.166667,Лофт,109.226891,111.94,1.272892,850.949683
4,2023,1166.666667,Лофт,107.708878,107.42,1.367340,853.238092
5,2024,1375.000000,Лофт,117.857143,109.52,1.497511,918.190318
6,2025,1735.000000,Лофт,126.181818,105.59,1.581222,1097.252806
7,2026,1983.333333,Лофт,114.313160,102.97,1.628184,1218.126018


Посвяты:


,Год,Средняя цена,Тип,индекс посвятов,ИПЦ,Накопленный_ИПЦ,реальная цена
0,2017,3000.000000,Посвят,100.000000,102.51,1.000000,3000.000000
1,2018,3200.000000,Посвят,106.666667,104.26,1.042600,3069.249952
2,2019,3600.000000,Посвят,112.500000,103.04,1.074295,3351.034740
3,2020,3750.000000,Посвят,104.166667,104.91,1.127043,3327.291190
4,2021,4100.000000,Посвят,109.333333,108.39,1.221602,3356.249071
5,2022,3800.000000,Посвят,92.682927,111.94,1.367461,2778.872495
6,2023,3950.000000,Посвят,103.947368,107.42,1.468927,2689.038196
7,2024,4100.000000,Посвят,103.797468,109.52,1.608769,2548.533209
8,2025,3733.333333,Посвят,91.056911,105.59,1.698699,2197.760777


In [ ]:
# лофты vs инфляция
try:
    loft_index
except NameError:
    raise NameError("Переменная 'loft_index' не найдена. Выполните предыдущие ячейки.")
if 'Год' not in loft_index.columns:
    loft_index = loft_index.reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=loft_index['Год'], y=loft_index['индекс лофтов'],
                         mode='lines+markers', name='Лофты',
                         line=dict(color='fuchsia', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Лофты</b>: %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Scatter(x=loft_index['Год'], y=loft_index['ИПЦ'],
                         mode='lines+markers', name='ИПЦ (инфляция)',
                         line=dict(color='gray', width=2, dash='dash'), marker=dict(size=8, symbol='square'),
                         hovertemplate='<b>Год</b>: %{x}<br><b>ИПЦ</b>: %{y:.1f}%<extra></extra>'))
fig.update_layout(title='Динамика цен на лофты vs официальная инфляция',
                  xaxis_title='Год', yaxis_title='Индекс, % (год к году)',
                  hovermode='x unified', template='plotly_white',
                  xaxis=dict(tickmode='linear', tick0=loft_index['Год'].min(), dtick=1))
fig.show()

In [ ]:
# посвяты vs инфляция
try:
    posvyat_index
except NameError:
    raise NameError("Переменная 'posvyat_index' не найдена. Выполните предыдущие ячейки.")

if 'Год' not in posvyat_index.columns:
    posvyat_index = posvyat_index.reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=posvyat_index['Год'], y=posvyat_index['индекс посвятов'],
                         mode='lines+markers', name='Посвяты',
                         line=dict(color='teal', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Посвяты</b>: %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Scatter(x=posvyat_index['Год'], y=posvyat_index['ИПЦ'],
                         mode='lines+markers', name='ИПЦ (инфляция)',
                         line=dict(color='gray', width=2, dash='dash'), marker=dict(size=8, symbol='square'),
                         hovertemplate='<b>Год</b>: %{x}<br><b>ИПЦ</b>: %{y:.1f}%<extra></extra>'))
fig.update_layout(title='Динамика цен на посвят vs официальная инфляция',
                  xaxis_title='Год', yaxis_title='Индекс, % (год к году)',
                  hovermode='x unified', template='plotly_white',
                  xaxis=dict(tickmode='linear', tick0=posvyat_index['Год'].min(), dtick=1))
fig.show()

In [ ]:
# Реальные цены лофтов + посвятов
try:
    loft_index, posvyat_index
except NameError:
    raise NameError("Переменные 'loft_index' и 'posvyat_index' не найдены. Выполните предыдущие ячейки.")
for df in (loft_index, posvyat_index):
    if 'Год' not in df.columns:
        df.reset_index(inplace=True)
fig = go.Figure()
fig.add_trace(go.Scatter(x=loft_index['Год'], y=loft_index['реальная цена'],
                         mode='lines+markers', name='Лофты',
                         line=dict(color='fuchsia', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Лофты (реальная)</b>: %{y:,.0f} ₽<extra></extra>'))
fig.add_trace(go.Scatter(x=posvyat_index['Год'], y=posvyat_index['реальная цена'],
                         mode='lines+markers', name='Посвяты',
                         line=dict(color='teal', width=3), marker=dict(size=10),
                         hovertemplate='<b>Год</b>: %{x}<br><b>Посвяты (реальная)</b>: %{y:,.0f} ₽<extra></extra>'))
fig.update_layout(title='Реальные цены лофтов и посвятов',
                  xaxis_title='Год', yaxis_title='Цена в ценах базового года, ₽',
                  hovermode='x unified', template='plotly_white',
                  xaxis=dict(tickmode='linear', dtick=1), legend=dict(x=0.02, y=0.98))
fig.show()

In [ ]:
# Средневзвешенная ключевая ставка ЦБ РФ по годам

df = pd.read_excel('../data/ключевая ставка.xlsx')
df.columns = ['дата', 'ставка']
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("✅ Загружено строк:", len(df))

In [ ]:
# Средневзвешенная ключевая ставка ЦБ РФ по годам
df['дата'] = pd.to_datetime(df['дата'], dayfirst=True, errors='coerce')
df['ставка'] = df['ставка'].astype(str).str.replace(',', '.').astype(float)
df = df.dropna(subset=['дата', 'ставка']).sort_values('дата').reset_index(drop=True)

df['год'] = df['дата'].dt.year
df['дата_конец'] = df['дата'].shift(-1)
df['дата_конец'] = df['дата_конец'].fillna(pd.Timestamp('2026-01-01'))
years = sorted([y for y in df['год'].unique() if y <= 2025])
results = []
for year in years:
    start = pd.Timestamp(f'{year}-01-01')
    end = pd.Timestamp(f'{year+1}-01-01')  # конец года – начало следующего
    mask = (df['дата'] < end) & (df['дата_конец'] > start)
    year_df = df.loc[mask].copy()
    year_df['начало'] = year_df['дата'].clip(lower=start)
    year_df['конец']  = year_df['дата_конец'].clip(upper=end)
    year_df['дней'] = (year_df['конец'] - year_df['начало']).dt.days
    weighted = (year_df['ставка'] * year_df['дней']).sum() / year_df['дней'].sum()
    results.append({'Год': year, 'Средневзвешенная ключевая ставка, %': round(weighted, 2)})
df_keyrate_calc = pd.DataFrame(results)

print("Средневзвешенная ключевая ставка ЦБ РФ по годам (рассчитано по дневным данным):")
print(df_keyrate_calc.to_string(index=False))

Средневзвешенная ключевая ставка ЦБ РФ по годам (рассчитано по дневным данным):
 Год  Средневзвешенная ключевая ставка, %
2017                                 9.11
2018                                 7.42
2019                                 7.33
2020                                 5.05
2021                                 5.75
2022                                10.62
2023                                 9.95
2024                                17.50
2025                                19.18


In [ ]:
# процентные изменения цен лофтов и посвятов

loft_growth = loft_yearly.sort_values('Год').reset_index(drop=True).copy()
posvyat_growth = posvyat_yearly.sort_values('Год').reset_index(drop=True).copy()

loft_growth['рост лофтов, %'] = loft_growth['Средняя цена'].pct_change() * 100
posvyat_growth['рост посвятов, %'] = posvyat_growth['Средняя цена'].pct_change() * 100

print("Лофты:")
display(loft_growth)
print("Посвяты:")
display(posvyat_growth)

Лофты:


,Год,Средняя цена,Тип,"рост лофтов, %"
0,2019,887.500000,Лофт,NaN
1,2020,875.000000,Лофт,-1.408451
2,2021,991.666667,Лофт,13.333333
3,2022,1083.166667,Лофт,9.226891
4,2023,1166.666667,Лофт,7.708878
5,2024,1375.000000,Лофт,17.857143
6,2025,1735.000000,Лофт,26.181818
7,2026,1983.333333,Лофт,14.313160


Посвяты:


,Год,Средняя цена,Тип,"рост посвятов, %"
0,2017,3000.000000,Посвят,NaN
1,2018,3200.000000,Посвят,6.666667
2,2019,3600.000000,Посвят,12.500000
3,2020,3750.000000,Посвят,4.166667
4,2021,4100.000000,Посвят,9.333333
5,2022,3800.000000,Посвят,-7.317073
6,2023,3950.000000,Посвят,3.947368
7,2024,4100.000000,Посвят,3.797468
8,2025,3733.333333,Посвят,-8.943089


In [ ]:
# Сводный график: темпы роста лофтов, посвятов и ключевая ставка

try:
    loft_growth
    posvyat_growth
except NameError:
    raise NameError("Переменные 'loft_growth' и 'posvyat_growth' не найдены. Выполните предыдущую ячейку.")
try:
    df_rate = df_keyrate_calc
except NameError:
    raise NameError("Переменная 'df_keyrate_calc' не найдена. Выполните ячейки с ключевой ставкой.")

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=loft_growth['Год'],
    y=loft_growth['рост лофтов, %'],
    mode='lines+markers',
    name='Рост цены лофтов, %',
    line=dict(color='fuchsia', width=3),
    marker=dict(size=9),
    hovertemplate='<b>Год</b>: %{x}<br><b>Рост лофтов</b>: %{y:.1f}%<extra></extra>'
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=posvyat_growth['Год'],
    y=posvyat_growth['рост посвятов, %'],
    mode='lines+markers',
    name='Рост цены посвятов, %',
    line=dict(color='teal', width=3),
    marker=dict(size=9),
    hovertemplate='<b>Год</b>: %{x}<br><b>Рост посвятов</b>: %{y:.1f}%<extra></extra>'
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_rate['Год'],
    y=df_rate['Средневзвешенная ключевая ставка, %'],
    mode='lines+markers',
    name='Ключевая ставка',
    line=dict(color='grey', width=3, dash='dot'),
    marker=dict(size=9, color='grey'),
    hovertemplate='<b>Год</b>: %{x}<br><b>Ключевая ставка</b>: %{y:.2f}%<extra></extra>'
), secondary_y=True)

fig.update_layout(
    title='Темпы роста цен лофтов и посвятов в сравнении с ключевой ставкой ЦБ РФ',
    xaxis_title='Год',
    hovermode='x unified',
    template='plotly_white',
    xaxis=dict(tickmode='linear', dtick=1),
    legend=dict(x=0.02, y=0.98)
)
fig.update_yaxes(title_text='Рост цены, %', secondary_y=False)
fig.update_yaxes(title_text='Ключевая ставка, %', secondary_y=True)

fig.show()